In [1]:
import os
import re
import math
import json
from tqdm import tqdm
import random
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
hf_token = os.getenv('HF_TOKEN')
pinecone_api_key = os.getenv('PINECONE_API_KEY')
index_name = "products-vectorstore"

In [3]:
login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
from items import Item

In [5]:
import joblib
train = joblib.load("train.pkl")

In [6]:
train[0].prompt

'How much does this cost to the nearest dollar?\n\nDelphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7\n\nPrice is $227.00'

In [7]:
from pinecone import Pinecone
pc = Pinecone(
    api_key=pinecone_api_key
)

In [13]:
assert index_name in [index.name for index in pc.list_indexes()], f"Index '{index_name}' does not exist. Please create it in the Pinecone console."
index = pc.Index(index_name)

In [9]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [10]:
vector = model.encode("My name is Bhavin")
len(vector)

384

In [20]:
vector.tolist()

[-0.0872148871421814,
 -0.06643718481063843,
 -0.026641368865966797,
 0.037724945694208145,
 -0.062439922243356705,
 -0.027304839342832565,
 0.13461709022521973,
 0.002570606768131256,
 0.03136121854186058,
 -0.04278689995408058,
 -0.04813641309738159,
 -0.1517936736345291,
 0.04961729794740677,
 -0.02256813645362854,
 0.006117488723248243,
 0.028088361024856567,
 0.0030007068999111652,
 0.04558054357767105,
 -0.07324938476085663,
 -0.09481627494096756,
 -0.09082001447677612,
 0.0947287529706955,
 -0.003761779982596636,
 -0.009456289000809193,
 -0.08751826733350754,
 0.0014873044565320015,
 0.044167350977659225,
 0.044284697622060776,
 -0.03659516200423241,
 -0.00489121163263917,
 -6.20265636825934e-05,
 -0.02598138153553009,
 0.04466714337468147,
 0.013790421187877655,
 -0.0031439601443707943,
 -0.008102970197796822,
 -0.14484988152980804,
 0.058580659329891205,
 0.011712648905813694,
 0.012619276531040668,
 -0.05565233156085014,
 -0.04245031252503395,
 -0.006837078370153904,
 -0.0366

In [14]:
import numpy as np
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def how_similar(text1, text2):
    vector1, vector2 = model.encode([text1, text2])
    similarity = cosine_similarity(vector1, vector2)
    print(f"Similarity between {text1} and {text2} is {similarity*100:.1f}%")
how_similar("Java", "C++")
how_similar("Java", "mug")
how_similar("Cup of Java", "mug")

Similarity between Java and C++ is 50.7%
Similarity between Java and mug is 25.8%
Similarity between Cup of Java and mug is 49.3%


In [15]:
def description(item):
    text = item.prompt.replace("How much does this cost to the nearest dollar?\n\n", "")
    return text.split("\n\nPrice is $")[0]

In [16]:
description(train[0])

'Delphi FG0166 Fuel Pump Module\nDelphi brings 80 years of OE Heritage into each Delphi pump, ensuring quality and fitment for each Delphi part. Part is validated, tested and matched to the right vehicle application Delphi brings 80 years of OE Heritage into each Delphi assembly, ensuring quality and fitment for each Delphi part Always be sure to check and clean fuel tank to avoid unnecessary returns Rigorous OE-testing ensures the pump can withstand extreme temperatures Brand Delphi, Fit Type Vehicle Specific Fit, Dimensions LxWxH 19.7 x 7.7 x 5.1 inches, Weight 2.2 Pounds, Auto Part Position Unknown, Operation Mode Mechanical, Manufacturer Delphi, Model FUEL PUMP, Dimensions 19.7'

## Now we populate our RAG datastore

The next cell populates the 400,000 items in Chroma.

Feel free to reduce the number of documents if this takes too long! You can change to:  
`NUMBER_OF_DOCUMENTS = 20000`  
And that's plenty for a perfectly good RAG pipeline.

Just note that if you interrupt the below cell while it's running, you might need to clear out the Chroma datastore (by rerunning the earlier cell that deletes the collection), before you run it again. Otherwise it will complain that there are existing documents with the same ID.

In [25]:
NUMBER_OF_DOCUMENTS = 20_000
train = random.sample(train, 20_000)
BATCH_SIZE = 500
for i in tqdm(range(0, NUMBER_OF_DOCUMENTS, BATCH_SIZE)):
    documents = [description(item) for item in train[i: i+BATCH_SIZE]]
    vectors = model.encode(documents).astype(float).tolist()
    metadatas = [{"category": item.category, "price": item.price,'description':documents[j]} for j, item in enumerate(train[i: i+BATCH_SIZE])]
    ids = [f"doc_{j}" for j in range(i, i+len(documents))]
    
    upsert = [
        {"id":id,
         "values":vector,
         "metadata":metadata,
         }
        for  id, vector, metadata in zip(ids, vectors, metadatas)
    ]
    
    index.upsert(upsert)

100%|██████████| 40/40 [17:00<00:00, 25.51s/it]
